### Loading Silver

In [0]:
%run /Workspace/Users/anupatil172006@gmail.com/Smart-Patient-Readmission-Risk-Pipeline/config/config.py

In [0]:
# ==========================================================
# Smart Patient Readmission Risk Pipeline
# Gold Layer - Configuration & Silver Load
# ==========================================================

from pyspark.sql import functions as F

print("Gold aggregation started.")
print(f"Database: {DATABASE}")

silver_df = spark.table(SILVER_ADMISSIONS)

print(f"Silver rows: {silver_df.count()}")

Gold aggregation started.
Database: dbacademy.smart_readmission
Silver rows: 692


### Readmission by Diagnosis

In [0]:
# ==========================================================
# Gold 1 - Readmission by Diagnosis
# ==========================================================

readmission_by_diagnosis = (
    silver_df
    .groupBy("diagnosis_category")
    .agg(
        F.count("*").alias("total_admissions"),

        F.sum(
            F.col("readmitted_within_30_days")
        ).alias("readmissions"),

        F.round(
            F.avg(
                F.col("readmitted_within_30_days")
            ) * 100,
            2
        ).alias("readmission_rate_pct"),

        F.round(
            F.avg("length_of_stay"),
            2
        ).alias("avg_length_of_stay")
    )
    .orderBy(
        F.desc("readmission_rate_pct")
    )
)

readmission_by_diagnosis.show(
    20,
    truncate=False
)

+------------------+----------------+------------+--------------------+------------------+
|diagnosis_category|total_admissions|readmissions|readmission_rate_pct|avg_length_of_stay|
+------------------+----------------+------------+--------------------+------------------+
|Respiratory       |115             |33          |28.7                |5.76              |
|Cardiovascular    |113             |32          |28.32               |5.67              |
|Nephrology        |46              |11          |23.91               |6.28              |
|Oncology          |122             |29          |23.77               |6.43              |
|Infectious        |53              |11          |20.75               |6.43              |
|Orthopedics       |58              |12          |20.69               |6.6               |
|Endocrine         |64              |13          |20.31               |3.81              |
|Neurology         |63              |11          |17.46               |6.92              |

### Writing gold table

In [0]:
# ==========================================================
# Write Gold - Readmission by Diagnosis
# ==========================================================

(
    readmission_by_diagnosis.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_READMISSION_BY_DIAGNOSIS)
)

print("✅ Gold table created successfully.")
print(GOLD_READMISSION_BY_DIAGNOSIS)

✅ Gold table created successfully.
dbacademy.smart_readmission.readmission_by_diagnosis


### Which departments have the highest workload and readmission rate?

In [0]:
# ==========================================================
# Gold 2 - Department Performance
# ==========================================================

department_performance = (
    silver_df
    .groupBy("department")
    .agg(
        F.count("*").alias("total_admissions"),

        F.sum(
            F.col("readmitted_within_30_days")
        ).alias("readmissions"),

        F.round(
            F.avg(
                F.col("readmitted_within_30_days")
            ) * 100,
            2
        ).alias("readmission_rate_pct"),

        F.round(
            F.avg("length_of_stay"),
            2
        ).alias("avg_length_of_stay"),

        F.round(
            F.avg("age"),
            1
        ).alias("avg_patient_age")
    )
    .orderBy(
        F.desc("readmission_rate_pct")
    )
)

department_performance.show(
    20,
    truncate=False
)

+----------------+----------------+------------+--------------------+------------------+---------------+
|department      |total_admissions|readmissions|readmission_rate_pct|avg_length_of_stay|avg_patient_age|
+----------------+----------------+------------+--------------------+------------------+---------------+
|ICU             |91              |29          |31.87               |10.69             |55.7           |
|Cardiology      |85              |25          |29.41               |4.91              |57.4           |
|Pulmonology     |80              |18          |22.5                |4.68              |54.8           |
|Oncology        |86              |19          |22.09               |7.31              |54.4           |
|Orthopedics     |53              |11          |20.75               |6.3               |57.9           |
|General Medicine|248             |43          |17.34               |3.95              |55.4           |
|Neurology       |49              |8           |16.33  

In [0]:
# ==========================================================
# Write Gold - Department Performance
# ==========================================================

(
    department_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_DEPARTMENT_PERFORMANCE)
)

print("✅ Department performance table created.")
print(GOLD_DEPARTMENT_PERFORMANCE)

✅ Department performance table created.
dbacademy.smart_readmission.department_performance


### Which patient age groups have the highest readmission risk

In [0]:
# ==========================================================
# Gold 3 - Age Group Risk
# ==========================================================

age_group_risk = (
    silver_df
    .groupBy("age_group")
    .agg(
        F.count("*").alias("total_admissions"),

        F.sum(
            F.col("readmitted_within_30_days")
        ).alias("readmissions"),

        F.round(
            F.avg(
                F.col("readmitted_within_30_days")
            ) * 100,
            2
        ).alias("readmission_rate_pct"),

        F.round(
            F.avg("length_of_stay"),
            2
        ).alias("avg_length_of_stay"),

        F.round(
            F.avg("prior_admission_count"),
            2
        ).alias("avg_prior_admissions")
    )
    .orderBy(
        F.desc("readmission_rate_pct")
    )
)

age_group_risk.show(
    20,
    truncate=False
)

+-----------+----------------+------------+--------------------+------------------+--------------------+
|age_group  |total_admissions|readmissions|readmission_rate_pct|avg_length_of_stay|avg_prior_admissions|
+-----------+----------------+------------+--------------------+------------------+--------------------+
|Elderly    |229             |69          |30.13               |5.63              |1.52                |
|Senior     |174             |39          |22.41               |5.89              |1.98                |
|Middle Age |166             |27          |16.27               |5.83              |1.54                |
|Young Adult|115             |17          |14.78               |5.83              |1.68                |
|Pediatric  |8               |1           |12.5                |5.88              |1.63                |
+-----------+----------------+------------+--------------------+------------------+--------------------+



### Writing Gold Table

In [0]:
# ==========================================================
# Write Gold - Age Group Risk
# ==========================================================

(
    age_group_risk.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_AGE_GROUP_RISK)
)

print("✅ Age group risk table created.")
print(GOLD_AGE_GROUP_RISK)

✅ Age group risk table created.
dbacademy.smart_readmission.age_group_risk


### Building Patirnt risk profile

In [0]:
# ==========================================================
# Gold 4 - Patient Risk Profile
# ==========================================================

patient_risk_profile = (
    silver_df
    .groupBy(
        "patient_id",
        "patient_name",
        "age",
        "age_group",
        "gender",
        "comorbidity_index"
    )
    .agg(
        F.count("*").alias("total_admissions"),

        F.sum(
            F.col("readmitted_within_30_days")
        ).alias("total_readmissions"),

        F.round(
            F.avg(
                F.col("readmitted_within_30_days")
            ) * 100,
            2
        ).alias("readmission_rate_pct"),

        F.round(
            F.avg("length_of_stay"),
            2
        ).alias("avg_length_of_stay"),

        F.max(
            "prior_admission_count"
        ).alias("prior_admission_count"),

        F.countDistinct(
            "diagnosis_category"
        ).alias("diagnosis_category_count")
    )
)


# ==========================================================
# Calculate Patient Risk Score
# ==========================================================

patient_risk_profile = (
    patient_risk_profile
    .withColumn(
        "risk_score",
        (
            # ------------------------------------------------
            # 1. Readmission History
            # Maximum contribution = 50 points
            # ------------------------------------------------
            (
                F.col("readmission_rate_pct") * F.lit(0.50)
            )

            +

            # ------------------------------------------------
            # 2. Prior Admissions
            # Maximum contribution = 20 points
            # ------------------------------------------------
            (
                F.least(
                    F.col("prior_admission_count") * F.lit(10),
                    F.lit(20)
                )
            )

            +

            # ------------------------------------------------
            # 3. Age Risk
            # Maximum contribution = 15 points
            # ------------------------------------------------
            (
                F.when(
                    F.col("age") >= 65,
                    F.lit(15)
                )
                .when(
                    F.col("age") >= 50,
                    F.lit(10)
                )
                .otherwise(
                    F.lit(0)
                )
            )

            +

            # ------------------------------------------------
            # 4. Comorbidity Proxy
            # Maximum contribution = 15 points
            # ------------------------------------------------
            (
                F.when(
                    F.col("comorbidity_index") == "High",
                    F.lit(15)
                )
                .when(
                    F.col("comorbidity_index") == "Moderate",
                    F.lit(8)
                )
                .otherwise(
                    F.lit(0)
                )
            )
        )
    )
)


# ==========================================================
# Risk Category
# ==========================================================

patient_risk_profile = (
    patient_risk_profile

    .withColumn(
        "risk_score",
        F.round(
            F.col("risk_score"),
            2
        )
    )

    .withColumn(
        "risk_category",
        F.when(
            F.col("risk_score") >= 50,
            "High"
        )
        .when(
            F.col("risk_score") >= 25,
            "Medium"
        )
        .otherwise(
            "Low"
        )
    )

    .orderBy(
        F.desc("risk_score")
    )
)


# ==========================================================
# Display Results
# ==========================================================

print("Patient Risk Profile:")

patient_risk_profile.show(
    20,
    truncate=False
)

Patient Risk Profile:
+----------+---------------------+---+----------+------+-----------------+----------------+------------------+--------------------+------------------+---------------------+------------------------+----------+-------------+
|patient_id|patient_name         |age|age_group |gender|comorbidity_index|total_admissions|total_readmissions|readmission_rate_pct|avg_length_of_stay|prior_admission_count|diagnosis_category_count|risk_score|risk_category|
+----------+---------------------+---+----------+------+-----------------+----------------+------------------+--------------------+------------------+---------------------+------------------------+----------+-------------+
|P00061    |Urvashi Ray          |69 |Elderly   |F     |High             |3               |2                 |66.67               |6.67              |2                    |3                       |83.34     |High         |
|P00031    |Theodore Devi        |85 |Elderly   |M     |High             |6           

In [0]:
# ==========================================================
# Write GOLD 4 - Patient Risk Profile
# ==========================================================

(
    patient_risk_profile.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_PATIENT_RISK_PROFILE)
)

print("✅ Patient risk profile table created.")
print(GOLD_PATIENT_RISK_PROFILE)

✅ Patient risk profile table created.
dbacademy.smart_readmission.patient_risk_profile


### Gold Validation

In [0]:
# ==========================================================
# GOLD LAYER - FINAL VALIDATION
# ==========================================================

print("================================================")
print("GOLD LAYER VALIDATION")
print("================================================")

gold_diagnosis = spark.table(
    GOLD_READMISSION_BY_DIAGNOSIS
)

gold_department = spark.table(
    GOLD_DEPARTMENT_PERFORMANCE
)

gold_age = spark.table(
    GOLD_AGE_GROUP_RISK
)

gold_patient = spark.table(
    GOLD_PATIENT_RISK_PROFILE
)

print(
    f"Readmission by Diagnosis rows: "
    f"{gold_diagnosis.count()}"
)

print(
    f"Department Performance rows: "
    f"{gold_department.count()}"
)

print(
    f"Age Group Risk rows: "
    f"{gold_age.count()}"
)

print(
    f"Patient Risk Profile rows: "
    f"{gold_patient.count()}"
)

GOLD LAYER VALIDATION
Readmission by Diagnosis rows: 9
Department Performance rows: 7
Age Group Risk rows: 5
Patient Risk Profile rows: 199


In [0]:
# ==========================================================
# Overall Hospital KPIs
# ==========================================================

overall_kpis = silver_df.agg(
    F.countDistinct("patient_id").alias("total_patients"),
    F.count("*").alias("total_admissions"),
    F.sum("readmitted_within_30_days").alias("total_readmissions"),
    F.round(
        F.avg("readmitted_within_30_days") * 100,
        2
    ).alias("overall_readmission_rate_pct"),
    F.round(
        F.avg("length_of_stay"),
        2
    ).alias("avg_length_of_stay")
)

overall_kpis.show()

+--------------+----------------+------------------+----------------------------+------------------+
|total_patients|total_admissions|total_readmissions|overall_readmission_rate_pct|avg_length_of_stay|
+--------------+----------------+------------------+----------------------------+------------------+
|           199|             692|               153|                       22.11|              5.78|
+--------------+----------------+------------------+----------------------------+------------------+



In [0]:
# ==========================================================
# Monthly Readmission Trend
# ==========================================================

monthly_readmission_trend = (
    silver_df
    .groupBy("admission_month")
    .agg(
        F.count("*").alias("total_admissions"),

        F.sum(
            "readmitted_within_30_days"
        ).alias("readmissions"),

        F.round(
            F.avg("readmitted_within_30_days") * 100,
            2
        ).alias("readmission_rate_pct")
    )
    .orderBy("admission_month")
)

monthly_readmission_trend.show(
    20,
    truncate=False
)

+---------------+----------------+------------+--------------------+
|admission_month|total_admissions|readmissions|readmission_rate_pct|
+---------------+----------------+------------+--------------------+
|2025-08        |42              |9           |21.43               |
|2025-09        |58              |15          |25.86               |
|2025-10        |68              |15          |22.06               |
|2025-11        |52              |10          |19.23               |
|2025-12        |57              |10          |17.54               |
|2026-01        |60              |13          |21.67               |
|2026-02        |50              |8           |16.0                |
|2026-03        |57              |12          |21.05               |
|2026-04        |53              |16          |30.19               |
|2026-05        |68              |21          |30.88               |
|2026-06        |60              |12          |20.0                |
|2026-07        |50              |

In [0]:
# ==========================================================
# Write Monthly Readmission Trend
# ==========================================================

monthly_readmission_trend.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{DATABASE}.monthly_readmission_trend"
    )

print("✅ Monthly readmission trend table created.")

✅ Monthly readmission trend table created.
